# YOLOv8 Classifier Training

In [1]:
!pip install ultralytics datasets huggingface_hub opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.2 MB/s eta 0:00:00


In [9]:
import os
import cv2
import hashlib
import shutil
import random
from pathlib import Path
from collections import Counter
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Isolate this dataset to prevent caching conflicts
DATA_DIR = Path("../datasets/classification_v3")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Load Base Detector to locate subjects for cropping
BASE_DETECTOR = YOLO("yolov8n.pt")

In [10]:
def extract_multiple_frames(video_path: str, base_output_path: Path, num_frames=3):
    """Extracts full video frames without cropping to preserve background context."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return

    fractions = [0.15, 0.30, 0.50, 0.70, 0.85]
    saved_count = 0

    for i, frac in enumerate(fractions):
        if saved_count >= num_frames:
            break

        frame_pos = int(total_frames * frac)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_pos)
        ret, frame = cap.read()
        if ret:
            out_name = f"{base_output_path.stem}_f{i}{base_output_path.suffix}"
            final_out = base_output_path.parent / out_name
            cv2.imwrite(str(final_out), frame)
            saved_count += 1

    cap.release()

In [11]:
def resolve_hf_cache_path(vid_url):
    if not isinstance(vid_url, str): return None
    if vid_url.startswith("hf://datasets/"):
        clean_url = vid_url[len("hf://datasets/"):]
        parts = clean_url.split('/')
        repo_id_with_rev = f"{parts[0]}/{parts[1]}"
        filename = "/".join(parts[2:])
        repo_id = repo_id_with_rev.split('@')[0]
        revision = repo_id_with_rev.split('@')[1] if '@' in repo_id_with_rev else None
        return hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset", revision=revision)
    return vid_url

def prep_dataset():
    print("Loading full dataset...")
    ds = load_dataset("Voxel51/Safe_and_Unsafe_Behaviours")

    for split in ds.keys():
        table = ds[split].data
        vid_urls = []
        if 'video' in table.column_names:
            for chunk in table.column('video').chunks:
                vid_urls.extend(chunk.field('path').to_pylist())
        elif 'path' in table.column_names:
            vid_urls = table.column('path').to_pylist()

        total_vids = len(vid_urls)
        print(f"Extracting {total_vids} videos for {split} split...")

        for i, vid_url in enumerate(vid_urls):
            if not vid_url: continue

            try:
                vid_path = resolve_hf_cache_path(vid_url)
            except Exception as e:
                continue

            if not vid_path: continue

            filename = Path(vid_path).name
            class_id = filename.split('_')[0]

            # Create a deterministic 80/20 train/val split
            is_val = int(hashlib.md5(vid_url.encode('utf-8')).hexdigest(), 16) % 5 == 0
            actual_split = 'val' if is_val else 'train'

            class_dir = DATA_DIR / actual_split / class_id
            class_dir.mkdir(parents=True, exist_ok=True)

            out_img_base = class_dir / f"{filename.split('.')[0]}.jpg"
            check_img = class_dir / f"{filename.split('.')[0]}_f0.jpg"

            if not check_img.exists():
                extract_multiple_frames(vid_path, out_img_base, num_frames=3)

    print("Dataset extraction complete.")

if not (DATA_DIR / "val").exists():
    prep_dataset()

Loading full dataset...


Resolving data files:   0%|          | 0/695 [00:00<?, ?it/s]

Extracting 691 videos for train split...
Dataset extraction complete.


In [12]:
def oversample_minority_classes(split="train", target_count=250):
    """Ensures every class in the training set has at least `target_count` samples."""
    split_dir = DATA_DIR / split
    if not split_dir.exists():
        return

    classes = [d for d in split_dir.iterdir() if d.is_dir()]

    print(f"\n--- Balancing {split.upper()} Dataset ---")
    for class_dir in classes:
        images = list(class_dir.glob("*.jpg"))
        current_count = len(images)

        if current_count == 0:
            continue

        if current_count < target_count:
            needed = target_count - current_count
            print(f"Class {class_dir.name}: {current_count} images -> Oversampling by {needed}")

            for i in range(needed):
                src = random.choice(images)
                dst = class_dir / f"{src.stem}_oversample_{i}{src.suffix}"
                shutil.copy2(src, dst)
        else:
            print(f"Class {class_dir.name}: {current_count} images -> OK")

oversample_minority_classes("train", target_count=250)


--- Balancing TRAIN Dataset ---
Class 0: 483 images -> OK
Class 3: 132 images -> Oversampling by 118
Class 7: 84 images -> Oversampling by 166
Class 6: 81 images -> Oversampling by 169
Class 1: 246 images -> Oversampling by 4
Class 5: 78 images -> Oversampling by 172
Class 2: 333 images -> OK
Class 4: 180 images -> Oversampling by 70


In [13]:
# Initialize YOLOv8 Nano Classifier
model = YOLO("yolov8n-cls.pt")

# Execute highly-regularized training loop
results = model.train(
    data=str(DATA_DIR.absolute()),
    epochs=25,
    imgsz=416,
    batch=16,
    project="../outputs",
    name="yolo_classifier_v3",
    device=0,

    # 1. Early Stopping
    patience=5,

    # 2. Strict Regularization
    dropout=0.3,
    weight_decay=0.001,

    # 3.  Data Augmentation
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=15.0,
    fliplr=0.5,    # Horizontal flip
    mixup=0.2      # Mixup images
)

Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/../datasets/classification_v3, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_classifier_v3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma

In [14]:
# Run final validation on the test/val split
metrics = model.val()
print(f"\nFINAL Top-1 Accuracy: {metrics.top1:.4f}")
print(f"FINAL Top-5 Accuracy: {metrics.top5:.4f}")

Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,445,128 parameters, 0 gradients, 3.3 GFLOPs
train: /datasets/classification_v3/train... found 2316 images in 8 classes ✅ 
val: /datasets/classification_v3/val... found 456 images in 8 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1827.3±1351.5 MB/s, size: 551.2 KB)
val: Scanning /datasets/classification_v3/val... 456 images, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 112.5Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 29/29 1.5it/s 19.9s
                   all      0.643      0.976
Speed: 1.1ms preprocess, 1.6ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val-2

FINAL Top-1 Accuracy: 0.6425
FINAL Top-5 Accuracy: 0.9759


In [16]:
from google.colab import files

# Path to the best model weights
weight_file = '/content/runs/outputs/yolo_classifier_v3/weights/best.pt'

# Download the file
files.download(weight_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>